# Flag Game: verified corpus and local RL

Frozen GPT-4o vision agents identify a country from private crops. Independent local actor–critics learn when to deposit and retrieve verified color observations.

**This is a protocol reconstruction, not an exact reproduction of the paper's scores.** The provided flag catalog and missing protocol details are documented choices. Read [the design and reproduction gaps](../docs/flag-game.md).

Paid execution is disabled by default. Configure `OPENAI_API_KEY` in the terminal **before starting Jupyter**; do not paste a secret into a saved cell.

In [ ]:
from pathlib import Path
import os
import json
import subprocess

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/flag-game.yaml").exists()
os.chdir(ROOT)
from millstlabs.flags.config import load_config
from millstlabs.flags.runner import plan, report, sized_config
from millstlabs.flags.task import catalog_images, generate_trial, crops_for

RUN = False
SIZE = "pilot"        # "smoke": one-round validation; "pilot": small screen; "full": 60 held-out trials.
BACKEND = "openai"   # "mock" is an offline wiring test, not an LLM.
MAX_USD = 10          # Cumulative ceiling for this output, including previous invocations.
HOURS = 12
cfg = load_config("configs/flag-game.yaml")
cfg.backend = BACKEND
cfg = sized_config(cfg, SIZE)
OUTPUT = ROOT / f"runs/flag-game{'-' + SIZE if SIZE != 'full' else ''}-{BACKEND}"
print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))
print("Output:", OUTPUT)
print(json.dumps(plan(cfg), indent=2))

## Inspect the task locally

The full flag below is **operator diagnostics**. Each vision agent receives only its own crop, without the target country or global crop position. These arrays never include metadata identifying the hidden country.

In [ ]:
import matplotlib.pyplot as plt
catalog, images = catalog_images(cfg.catalog, cfg)
trial = generate_trial(42, cfg, list(images))
crops = crops_for(trial, images, cfg)
fig, axes = plt.subplots(1, cfg.population + 1, figsize=(15, 2.3))
axes[0].imshow(images[trial.country], interpolation="nearest")
axes[0].set_title("Operator only\n" + trial.country)
for i, crop in enumerate(crops):
    axes[i+1].imshow(crop, interpolation="nearest")
    axes[i+1].set_title(f"Agent {i}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

## Launch or resume

The cumulative dollar ceiling may stop the run before completion. Increasing it explicitly permits more total spending; it is separate from the API account balance. No GPU is needed. Re-running with identical configuration/assets resumes, reusing completed API responses.

The five conditions are the reconstructed paper protocol, frozen private/shared corpus controllers, and RL-trained private/shared controllers. A sixth evaluation removes peer access from the trained shared controller without changing its weights.

In [ ]:
if RUN:
    if BACKEND == "openai" and not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Configure OPENAI_API_KEY in the terminal and restart Jupyter from that terminal.")
    command = ["bash", "scripts/flag_game.sh", "--mode", "run", "--size", SIZE,
               "--backend", BACKEND, "--output", str(OUTPUT), "--hours", str(HOURS)]
    if BACKEND == "openai":
        command += ["--max-usd", str(MAX_USD)]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Execution disabled. Inspect the plan before setting RUN=True.")

In [ ]:
results = None
if (OUTPUT / "latest.pt").exists():
    results = report(OUTPUT)
    print("Status:", results["status"], results["completed_jobs"], "/", results["total_jobs"])
    print("Cost guard charged/reserved USD:", results["cost_guard_charged_usd"])
    for arm, values in results["summaries"].items():
        print(arm, "trials:", values["trials"], "truth mass:", values.get("terminal_truth_mass"))
    for name, contrast in results["comparisons"].items():
        print(name, {k: contrast[k] for k in ["paired_trials", "mean_truth_mass_difference", "trial_bootstrap_95_interval"]})
    print(results["interpretation"])
else:
    print("No saved run here yet.")

In [ ]:
if results:
    values = [(a, r) for a, r in results["summaries"].items() if r["trials"]]
    if values:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for ax, metric, title in zip(axes, ["terminal_truth_mass", "social_uplift"],
                                     ["Terminal truth mass", "Social uplift over isolated crop guesses"]):
            ax.bar([a for a, _ in values], [r[metric] for _, r in values])
            ax.set_title(title)
            ax.tick_params(axis="x", rotation=35)
        fig.tight_layout()
        plt.show()

## Interpretation

Use **shared RL minus private RL** to assess peer access, **shared RL minus shared frozen** to assess learning, and the **same trained policy with private access** to assess whether it relies on peers. Initial majority-vote accuracy is a separate baseline. Wrong consensus is a failure, regardless of confidence.

Controller training receives correctness rewards after the trial; neither GPT nor its controller receives the target country as input. Imported colors measure information transfer, not new external discoveries. One training seed and six pilot trials cannot establish a robust advantage. Mock scores cannot establish any LLM result.